# 06 — Deep Blue data: single-time-point TIA estimates

The project relies on one post-therapy SPECT to refine the prediction for the individual patient. Before using a network for that step, this notebook measures how well the classical single-time-point (STP) methods recover the time-integrated activity (TIA) of notebook 02 when only one of the four scans is available.

For each structure (kidneys, spleen, healthy liver, lesions) and each scan used alone, with $A(t)$ the activity measured in that scan:

| Method | TIA from one measurement | Population information |
|---|---|---|
| (1) mono, population $T_\mathrm{eff}$ | $A(t)\, e^{\lambda t} / \lambda$, $\lambda = \ln 2 / T_\mathrm{eff}$ | effective half-life |
| (2) Hänscheid | $A(t)\, 2t / \ln 2$ | none |
| (3) uptake-washout, population shape | $A(t) \cdot \dfrac{1/\lambda_w - 1/\lambda_u}{e^{-\lambda_w t} - e^{-\lambda_u t}}$ | washout rate $\lambda_w$ and uptake rate $\lambda_u$ |

Method (3) uses the model of notebook 02, $A(t) = A_0 (e^{-\lambda_w t} - e^{-\lambda_u t})$, with the shape fixed by the population and the amplitude $A_0$ rescaled to the single measured point. All TIAs are integrals from 0 to infinity, in MBq·h.

Population kinetics come from two sources, kept separate:
- **Literature** (section 2), with the exact paper and location of every number.
- **Leave-one-patient-out (LOPO)** from the notebook 02 fits: for each patient, the population is the other patient. With only two patients this is a population of one, so the LOPO results measure transfer between two real patients rather than a population average.

The reference is the 4-time-point uptake-washout TIA of notebook 02 with its Monte Carlo 68% interval (10% random error per point plus the partial-volume term for lesions). The single measured point is also one of the four points of the reference fit, as in every STP validation study.

**Steps**
1. Inputs: activities, reference TIA and interval, notebook 02 fits.
2. Literature population kinetics.
3. Leave-one-patient-out population kinetics.
4. STP estimates and errors.
5. Error vs measurement time by structure type.
6. Saving.

**About this notebook.** Single-time-point TIA methods against the 4-point reference.
- **Inputs:** nb02 outputs.
- **Outputs:** `data/processed/deepblue/kinetics/stp_*`.
- **Run time:** < 1 min.
- **Environment:** .venv.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
PROC = WS / "data" / "processed" / "deepblue"
KIN = PROC / "kinetics"
PATIENTS = ["patient_4", "patient_6"]
SNMMI_LABEL = {"patient_4": "A", "patient_6": "B"}
SIGMA_POINT = "10%"

TYPES = ["kidney", "spleen", "liver", "lesion"]
STRUCTURE_TYPE = {"ltkidney": "kidney", "rtkidney": "kidney", "spleen": "spleen", "liver_healthy": "liver"}


def structure_type(name):
    return "lesion" if name.startswith("lesion") else STRUCTURE_TYPE.get(name)


half_lives = {json.loads((PROC / p / "timing.json").read_text())["half_life_s"] for p in PATIENTS}
assert len(half_lives) == 1
T_PHYS_H = half_lives.pop() / 3600.0
LAM_PHYS = np.log(2) / T_PHYS_H

plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6})
print(f"T1/2 = {T_PHYS_H:.1f} h, lambda_phys = {LAM_PHYS:.5f} 1/h")

## 1. Inputs

- Activities $A(t)$: `kinetics/tac.csv` (MBq, physical activity at the acquisition time). Healthy liver is the scan-1 reference liver without lesions, as in the SNMMI challenge; `liver_whole` and `liver_whole_perscan` are not used here.
- Reference: uptake-washout TIA of notebook 02 (`fits_all_models.csv`, `used == True`) and its 68% interval from `tia_uncertainty_mc.csv` (10% scenario).
- Notebook 02 fits of both models (mono and uptake-washout) per structure, used for the LOPO populations: $T_\mathrm{eff}$ of the mono fit, and $\lambda_w = \ln 2 / T_\mathrm{eff}$, $\lambda_u = \ln 2 / T_\mathrm{uptake}$ of the uptake-washout fit.

In [ ]:
tac = pd.read_csv(KIN / "tac.csv")
tac["type"] = tac.structure.map(structure_type)
tac = tac[tac.type.notna()].copy()

fits = pd.read_csv(KIN / "fits_all_models.csv")
fits = fits[fits.structure.map(structure_type).notna()]
mono = fits[fits.model == "mono"].set_index(["patient", "structure"])
upt = fits[fits.model == "uptake"].set_index(["patient", "structure"])
assert upt.used.all()

unc = pd.read_csv(KIN / "tia_uncertainty_mc.csv")
unc = unc[unc.sigma_point == SIGMA_POINT].set_index(["patient", "structure"])

ref = upt[["TIA_MBq_h", "T_eff_h", "T_uptake_h", "lam_bio_at_bound", "tail_fraction"]].rename(
    columns={"TIA_MBq_h": "TIA_ref"}).join(unc[["TIA_p16", "TIA_p84"]])
ref["type"] = [structure_type(s) for _, s in ref.index]
ref["T_eff_mono_h"] = mono.T_eff_h
ref["rel_p16"] = ref.TIA_p16 / ref.TIA_ref - 1
ref["rel_p84"] = ref.TIA_p84 / ref.TIA_ref - 1
ref.round(3)

## 2. Literature population kinetics (177Lu-DOTATATE)

**Method (1), effective half-life.** Hänscheid H, Lapa C, Buck AK, Lassmann M, Werner RA. *Dose mapping after endoradiotherapy with 177Lu-DOTATATE/DOTATOC by a single measurement after 4 days.* J Nucl Med 2018;59:75-81 (doi:10.2967/jnumed.117.193706). Results, section "Normalized Uptake Functions", text (not a table): "the tissue-specific effective half-life of the main component in all evaluated tissues was a median of 51 h (range, 40–68 h with 1 outlier at 106 h) in kidneys, 67 h (range, 55–117 h) in livers, 68 h (range, 52–99 h) in spleens, and 77 h (range, 56–130 h) in tumors". Caveats: 29 patients, 177Lu-DOTATATE (n = 22) and DOTATOC (n = 7); planar whole-body imaging without attenuation or scatter correction (only relative kinetics); main component of bi-exponential fits; tumours = NET lesions only (meningiomas excluded); whole liver, not healthy liver.

**Method (2)** is the approximation proposed in the same paper (Eq. 6): $\tilde A \approx A(t_l)\, 2 t_l / \ln 2$. It needs no population value. For a mono-exponential curve the ratio of the exact factor $e^{\lambda t}/\lambda$ to $2t/\ln 2$ is $2^x/(2x)$ with $x = t/T_\mathrm{eff}$: exact at $x = 1$ and $x = 2$, at most 6% below between them, and within ±10% for $0.78 \le x \le 2.40$ (computed here). The paper states "less than 10% deviation" for $0.75 < t/T_\mathrm{eff} < 2.5$ (Methods, repeated in the Discussion); the strict ratio is 1.12 at 0.75 and 1.13 at 2.5, so the narrower window is ours. The paper's own validation at $t_l = 96$ h gave median deviations of +5% (kidneys), +6% (livers), +8% (spleens) and +6% (NET lesions) (abstract).

**Method (3), uptake-washout shape.** The only DOTATATE source found with per-structure parameters of the same model is for **kidneys**: Devasia TP, Dewaraja YK, Frey KA, Wong KK, Schipper MJ. *A novel time-activity information-sharing approach using nonlinear mixed models for patient-specific dosimetry with reduced imaging time points: application in SPECT/CT after 177Lu-DOTATATE.* J Nucl Med 2021;62:1118-1125 (doi:10.2967/jnumed.120.256255). Eq. 1: $A(t) = c\,\frac{k_e k_a}{k_a - k_e}\,(e^{-k_e t} - e^{-k_a t})$, i.e. the same shape as notebook 02 with $\lambda_w = k_e$ (effective, includes physical decay) and $\lambda_u = k_a$. Individual fits for 20 kidneys of 10 patients are in **Supplemental Table 1** (Siemens Intevo, xSPECT Quant, University of Michigan, 2018-2020). The Results of the paper evaluate 18 clinical kidneys without saying which two are left out, so the set used here is not the cohort of the paper. The population shape used here is the median $k_a$ and median $k_e$ over the kidneys retained (see the overlap check below). For comparison only, Supplemental Table 2 lists the fixed effects used to simulate the virtual cohort, $k_a = 0.667$ h⁻¹, $k_e = 0.012$ h⁻¹.

No published DOTATATE population parameters of an uptake-washout curve were found for liver, spleen or lesions, so method (3) with literature kinetics is **not computed** for them. The search covered J Nucl Med and EJNMMI Physics through web search; the NETTER-1 dosimetry substudy (J Nucl Med 2025;66:449) and the HA-DOTATATE population PK model (EJNMMI Phys 2023, doi:10.1186/s40658-023-00565-4) appeared but were not checked in full text and are not used.

**Overlap with the Deep Blue patients.** Devasia et al. imaged volunteers at the University of Michigan in 2018-2020, the same centre, camera and reconstruction as the Deep Blue / SNMMI challenge data. Their patient 6 matches our patient 6 (SNMMI "B"): injected 7.34 vs 7.31 GBq, scan times later by a near-constant 0.44-0.69 h (consistent with a different time origin), and the same kidney pattern (right kidney 2.6-2.8 times the left in their data, 2.3-2.9 in ours, with the right kidney increasing from the first to the second scan in both). The paper's Discussion also names the images it deposited on Deep Blue: "patients 4 and 6 in the current study". **Their patient 4 is therefore our patient 4, at another cycle:** our series are cycle 2 (`SeriesDescription` `C2S1D0`...`C2S4D5`) and the paper used one of the cycles, which explains the different scan schedule (4.7, 98, 126, 175 h against our 3.7, 28, 103, 124 h). The right/left ratio is compatible (0.92-0.96 against 0.85-0.95) and the first-scan kidney activities agree (ratios 0.99 and 1.02). **Both kidney pairs are excluded from the literature population**, otherwise patients 4 and 6 would be predicted partly from their own kinetics. Their patient 2 has similar injected activity and scan days as our patient 4 but does not match (time differences 0.0, -3.0, -2.1, -3.7 h; right/left kidney ratio 1.19 in their data vs 0.85-0.95 in ours) and is kept.

In [ ]:
HAENSCHEID_TEFF_H = {"kidney": 51.0, "liver": 67.0, "spleen": 68.0, "lesion": 77.0}

DEVASIA_SUPPL_T1 = pd.DataFrame([
    (1, [3.8, 120.1], "L", [83.3, 22.0], 0.488, 0.013), (1, [3.8, 120.1], "R", [83.2, 21.1], 0.460, 0.014),
    (2, [3.7, 24.7, 101.0, 120.3], "L", [65.4, 56.8, 19.2, 14.3], 0.535, 0.014),
    (2, [3.7, 24.7, 101.0, 120.3], "R", [77.7, 67.8, 22.9, 16.9], 0.526, 0.014),
    (3, [4.9, 101.6, 127.5, 172.7], "L", [83.0, 20.2, 8.9, 9.3], 2.817, 0.015),
    (3, [4.9, 101.6, 127.5, 172.7], "R", [135.1, 21.2, 8.8, 9.2], 3.173, 0.020),
    (4, [4.7, 98.3, 125.9, 174.6], "L", [104.6, 25.6, 17.6, 10.2], 3.684, 0.015),
    (4, [4.7, 98.3, 125.9, 174.6], "R", [96.6, 23.6, 16.6, 9.8], 3.855, 0.015),
    (5, [4.4, 29.2, 48.4, 171.0], "L", [57.6, 43.8, 37.5, 9.5], 1.330, 0.010),
    (5, [4.4, 29.2, 48.4, 171.0], "R", [58.7, 40.1, 34.0, 9.5], 4.658, 0.012),
    (6, [4.1, 33.3, 100.1, 193.9], "L", [43.4, 46.8, 17.4, 5.4], 0.244, 0.014),
    (6, [4.1, 33.3, 100.1, 193.9], "R", [113.2, 126.5, 48.5, 15.2], 0.234, 0.014),
    (7, [4.1, 30.2, 98.2, 148.3], "L", [40.8, 31.5, 16.1, 9.7], 1.925, 0.010),
    (7, [4.1, 30.2, 98.2, 148.3], "R", [45.8, 37.1, 19.5, 11.9], 0.802, 0.010),
    (8, [3.8, 51.5, 123.3, 147.4], "L", [77.5, 42.9, 16.1, 11.0], 0.713, 0.014),
    (8, [3.8, 51.5, 123.3, 147.4], "R", [75.6, 40.7, 15.1, 11.3], 0.970, 0.014),
    (9, [3.8, 123.4], "L", [73.7, 21.7], 0.462, 0.012), (9, [3.8, 123.4], "R", [77.1, 27.7], 0.467, 0.010),
    (10, [3.8, 50.1, 98.2, 166.6], "L", [26.0, 33.1, 23.7, 14.6], 0.227, 0.007),
    (10, [3.8, 50.1, 98.2, 166.6], "R", [26.4, 31.3, 24.1, 15.6], 0.275, 0.006),
], columns=["devasia_patient", "t_h", "kidney", "activity_MBq", "ka_per_h", "ke_per_h"])

ours = tac[tac.structure.isin(["ltkidney", "rtkidney"])].pivot_table(index=["patient", "scan"], columns="structure",
                                                                      values="activity_MBq")
times = tac.groupby(["patient", "scan"]).t_h.first()
for dp, patient in [(6, "patient_6"), (4, "patient_4"), (2, "patient_4")]:
    d = DEVASIA_SUPPL_T1[DEVASIA_SUPPL_T1.devasia_patient == dp].set_index("kidney")
    t_ours = times.loc[patient].to_numpy()
    print(f"Devasia patient {dp} vs {patient}: time difference (h) {np.round(np.array(d.t_h['L']) - t_ours, 2)}")
    for side, col in [("L", "ltkidney"), ("R", "rtkidney")]:
        print(f"   {side} kidney activity, ours / Devasia: {np.round(ours.loc[patient, col].to_numpy() / np.array(d.activity_MBq[side]), 2)}")
    print(f"   R/L ratio Devasia {np.round(np.array(d.activity_MBq['R']) / np.array(d.activity_MBq['L']), 2)}, "
          f"ours {np.round((ours.loc[patient, 'rtkidney'] / ours.loc[patient, 'ltkidney']).to_numpy(), 2)}")

devasia_used = DEVASIA_SUPPL_T1[~DEVASIA_SUPPL_T1.devasia_patient.isin([4, 6])]
LIT_KIDNEY_SHAPE = {"lam_w": float(devasia_used.ke_per_h.median()), "lam_u": float(devasia_used.ka_per_h.median())}
print(f"\nDevasia kidneys used: {len(devasia_used)}; median ke = {LIT_KIDNEY_SHAPE['lam_w']:.4f} 1/h "
      f"(T_eff {np.log(2) / LIT_KIDNEY_SHAPE['lam_w']:.1f} h), median ka = {LIT_KIDNEY_SHAPE['lam_u']:.3f} 1/h "
      f"(T_uptake {np.log(2) / LIT_KIDNEY_SHAPE['lam_u']:.2f} h)")

The numbers support the overlap: for patient 6 the time offset is nearly constant and the right/left ratio is similar at every scan; our activities are 0.71-0.88 of theirs (different VOIs: Devasia et al. drew the kidneys on the first CT and propagated them by contour-guided alignment, our values use the challenge masks of each scan). For their patient 4 only the first scan is comparable in time: the later rows pair different days, so those activity ratios and time differences mean nothing. For patient 2 the right/left ratio is inverted (1.19 vs 0.85-0.95), so it is a different patient. The 16 kidneys kept include patients 1 and 9, whose fits rest on two time points only.

Excluding patient 4 as well moves the median $k_e$ from 0.0135 to 0.0125 h⁻¹ ($T_\mathrm{eff}$ 51.3 to 55.5 h) and the median $k_a$ from 0.758 to 0.624 h⁻¹ ($T_\mathrm{uptake}$ 0.92 to 1.11 h).

## 3. Leave-one-patient-out population kinetics

For each patient and structure type, the population values are the medians over the structures of that type in the **other** patient:
- method (1): $T_\mathrm{eff}$ of the notebook 02 mono-exponential fit (the model that method (1) assumes);
- method (3): $\lambda_w$ and $\lambda_u$ of the notebook 02 uptake-washout fit (medians of the rates).

Kidneys contribute two structures per patient, lesions two (patient 4) or four (patient 6), healthy liver one. Patient 6 has no spleen mask, so there is no LOPO spleen value for patient 4 and no spleen at all for patient 6: the spleen is evaluated only for patient 4 with literature kinetics and Hänscheid.

In [ ]:
rates = ref.assign(lam_w=np.log(2) / ref.T_eff_h, lam_u=np.log(2) / ref.T_uptake_h)

pop = []
for patient in PATIENTS:
    other = rates.loc[[p for p in PATIENTS if p != patient]]
    for typ in TYPES:
        o = other[other.type == typ]
        pop.append({"for_patient": patient, "source": "LOPO", "type": typ, "n_structures": len(o),
                    "T_eff_mono_h": float(o.T_eff_mono_h.median()),
                    "lam_w": float(o.lam_w.median()), "lam_u": float(o.lam_u.median())})
        lit = {"for_patient": patient, "source": "literature", "type": typ, "T_eff_mono_h": HAENSCHEID_TEFF_H[typ],
               "lam_w": np.nan, "lam_u": np.nan, "n_structures": np.nan}
        if typ == "kidney":
            lit.update(LIT_KIDNEY_SHAPE, n_structures=len(devasia_used))
        pop.append(lit)
pop = pd.DataFrame(pop)
pop["T_eff_uptake_h"] = np.log(2) / pop.lam_w
pop["T_uptake_h"] = np.log(2) / pop.lam_u
pop.set_index(["for_patient", "source", "type"])[["n_structures", "T_eff_mono_h", "T_eff_uptake_h", "T_uptake_h"]].round(2)

## 4. STP estimates and errors

Error = STP TIA / reference TIA − 1. "In 68%" means that the STP estimate falls inside the 68% interval of the 4-point reference, i.e. the single-point method is as good as the 4-point fit within its own uncertainty.

In [ ]:
def tia_mono(a, t, t_eff):
    lam = np.log(2) / t_eff
    return a * np.exp(lam * t) / lam


def tia_haenscheid(a, t):
    return a * 2 * t / np.log(2)


def tia_uptake(a, t, lam_w, lam_u):
    return a * (1 / lam_w - 1 / lam_u) / (np.exp(-lam_w * t) - np.exp(-lam_u * t))


popi = pop.set_index(["for_patient", "source", "type"])
rows = []
for r in tac.itertuples():
    base = {"patient": r.patient, "snmmi": SNMMI_LABEL[r.patient], "structure": r.structure, "type": r.type,
            "scan": r.scan, "t_h": r.t_h, "A_MBq": r.activity_MBq}
    rows.append({**base, "method": "Hänscheid 2t/ln2", "source": "none", "TIA_stp": tia_haenscheid(r.activity_MBq, r.t_h)})
    for source in ["literature", "LOPO"]:
        p = popi.loc[(r.patient, source, r.type)]
        if not np.isfinite(p.T_eff_mono_h):
            continue
        rows.append({**base, "method": "mono, population T_eff", "source": source,
                     "TIA_stp": tia_mono(r.activity_MBq, r.t_h, p.T_eff_mono_h)})
        if np.isfinite(p.lam_w):
            rows.append({**base, "method": "uptake-washout, population shape", "source": source,
                         "TIA_stp": tia_uptake(r.activity_MBq, r.t_h, p.lam_w, p.lam_u)})
stp = pd.DataFrame(rows).join(ref[["TIA_ref", "TIA_p16", "TIA_p84", "T_eff_h"]], on=["patient", "structure"])
stp["error"] = stp.TIA_stp / stp.TIA_ref - 1
stp["in_68"] = stp.TIA_stp.between(stp.TIA_p16, stp.TIA_p84)
stp["t_over_T_eff_ref"] = stp.t_h / stp.T_eff_h
stp["label"] = stp.method + " (" + stp.source + ")"
LABELS = ["mono, population T_eff (literature)", "mono, population T_eff (LOPO)", "Hänscheid 2t/ln2 (none)",
          "uptake-washout, population shape (literature)", "uptake-washout, population shape (LOPO)"]

table = stp.pivot_table(index=["patient", "structure", "scan"], columns="label", values="error")[LABELS]
(100 * table).round(1)

In [ ]:
def summary(g):
    e = g.error.abs()
    return pd.Series({"n": len(g), "median |error| %": 100 * e.median(), "max |error| %": 100 * e.max(),
                      "median error %": 100 * g.error.median(), "in 68%": f"{g.in_68.sum()}/{len(g)}"})


by_scan = stp.groupby(["label", "scan"]).apply(summary, include_groups=False).unstack("scan")
by_scan = by_scan.reindex(LABELS)
by_scan[["median |error| %", "max |error| %", "in 68%"]].round(1)

In [ ]:
by_type = stp[stp.scan.isin(["scan3", "scan4"])].groupby(["label", "type"]).apply(summary, include_groups=False)
by_type.reindex(LABELS, level=0).round(1)

The table above restricts to the late scans (scan 3 at about 100 h, scan 4 at 124 h for patient 4 and 193 h for patient 6), the only window where STP is expected to work.

## 5. Error vs measurement time by structure type

One column per structure type, one row per method. Colour = source of the population kinetics; marker = patient; lines join the scans of the same structure; filled markers are inside the 68% interval of the reference. The grey band is the 68% interval of the reference relative to its value (median over the structures of the type), the scale against which an STP error should be judged. The y-axis is clipped at ±60%; points outside are drawn on the edge as triangles.

In [ ]:
COLORS = {"literature": "#2a78d6", "LOPO": "#eb6834", "none": "#2a78d6"}
MARKERS = {"patient_4": "o", "patient_6": "s"}
METHODS = ["mono, population T_eff", "Hänscheid 2t/ln2", "uptake-washout, population shape"]
CLIP = 0.6

fig, axes = plt.subplots(len(METHODS), len(TYPES), figsize=(14, 9.5), sharex=True, sharey=True)
for j, typ in enumerate(TYPES):
    rt = ref[ref.type == typ]
    for i, method in enumerate(METHODS):
        ax = axes[i, j]
        ax.axhspan(100 * rt.rel_p16.median(), 100 * rt.rel_p84.median(), color="#d9d8d3", alpha=0.6, lw=0)
        ax.axhline(0, color="#52514e", lw=0.8)
        sub = stp[(stp.type == typ) & (stp.method == method)]
        notes = []
        if method.startswith("uptake") and typ != "kidney":
            notes.append("no literature shape")
        if method.startswith("mono") and typ == "spleen":
            notes.append("no LOPO (patient 6 has no spleen)")
        if notes:
            ax.text(0.02, 0.97, "\n".join(notes), transform=ax.transAxes, ha="left", va="top", color="#52514e", fontsize=8.5)
        for (source, patient, structure), g in sub.groupby(["source", "patient", "structure"]):
            g = g.sort_values("t_h")
            y = 100 * g.error.clip(-CLIP, CLIP)
            c = COLORS[source]
            ax.plot(g.t_h, y, color=c, lw=1.2, alpha=0.7)
            out = g.error.abs() > CLIP
            for (t, yy, inside, o) in zip(g.t_h, y, g.in_68, out):
                ax.plot(t, yy, marker=("^" if yy > 0 else "v") if o else MARKERS[patient], ms=7, color=c,
                        mfc=c if inside else "#fcfcfb", mew=1.5, ls="none")
        if i == 0:
            ax.set_title(typ, loc="left")
        if j == 0:
            ax.set_ylabel(f"{method}\nerror vs 4-point TIA (%)", fontsize=9)
        if i == len(METHODS) - 1:
            ax.set_xlabel("time of the single scan (h)")
        ax.set_ylim(-100 * CLIP - 5, 100 * CLIP + 5)

from matplotlib.lines import Line2D
handles = [Line2D([], [], color=COLORS["literature"], lw=2, label="literature kinetics (Hänscheid: no population)"),
           Line2D([], [], color=COLORS["LOPO"], lw=2, label="leave-one-patient-out kinetics"),
           Line2D([], [], color="#52514e", marker="o", ls="none", label="patient 4 (A)"),
           Line2D([], [], color="#52514e", marker="s", ls="none", label="patient 6 (B)"),
           Line2D([], [], color="#52514e", marker="o", mfc="#fcfcfb", ls="none", label="outside reference 68%"),
           plt.Rectangle((0, 0), 1, 1, color="#d9d8d3", label="reference 68% interval (median)")]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Single-time-point TIA: error vs time of the measurement", x=0.01, ha="left")
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(KIN / "stp_error_vs_time.png", dpi=150, bbox_inches="tight")
plt.show()

## Findings

Numbers from the tables above (13 structures: 12 for LOPO mono and uptake, because patient 4's spleen has no LOPO value; 4 kidneys for the literature uptake-washout shape).

1. **Scans at about 4 h and 30 h are unusable for STP with any method.** Hänscheid underestimates by 85-98% at 4 h and by 12-70% at about 30 h; the population methods have median |error| 15-47% at 4 h and 8-21% at 30 h, with outliers up to 80-140%. At these times the curve is still in the uptake or early-washout phase, which a single point cannot constrain.
2. **The scan at about 100 h (scan 3) is the best single time point for every method.** Literature mono $T_\mathrm{eff}$ and Hänscheid: median |error| 2.7% (max 18.8% and 14.9%), inside the reference 68% interval for 12/13 and 11/13 structures. Literature uptake-washout shape for kidneys: median 6.0%, max 7.9%. This agrees with the 96 h optimum of Hänscheid et al. and with their median deviations of +5% to +8%.
3. **Late scans fail when they leave the validity window of the approximation.** For patient 6 the last scan is at 193 h, i.e. 3.6 $T_\mathrm{eff}$ for the kidneys: Hänscheid underestimates by 36-41% (kidneys) and 23-28% (lesions 3-4), as expected for $t/T_\mathrm{eff} > 2.4$. For patient 4 the last scan (124 h, $t/T_\mathrm{eff}$ = 2.6 for the kidneys) gives -11% and -17%.
4. **Leave-one-patient-out kinetics are worse than the literature values** (median |error| at 100 h 10.2% mono, 9.2% uptake-washout; max 41.8% and 26.1%). Organs transfer well between the two patients (liver ≤ 4.4%, kidneys ≤ 15%), lesions do not: both lesions of patient 4 have $\lambda_\mathrm{bio}$ at the bound in the mono fit ($T_\mathrm{eff} = T_\mathrm{phys}$ = 159.5 h), so patient 6's lesions are predicted with a far too long half-life (+23% to +42%). One patient is not a population, and lesion kinetics differ between patients more than organ kinetics.
5. **The literature uptake-washout shape does not beat the literature mono $T_\mathrm{eff}$ for kidneys.** At 100 h: median 6.0% vs 2.2% (max 7.9% vs 12.9%). At about 30 h: median 9.2% vs 5.6%, max 10.6% vs 7.6%. The marginal gain seen at about 30 h before Devasia patient 4 was excluded (max 6.0%) came from having our patient 4 in its own population.
6. **The residual misses at 100 h are shared by all methods**: patient 4 lesion2 (-14% to -21%) and patient 6 left kidney (+8% to +15%). When all methods agree, the discrepancy is in the 4-point reference rather than in the STP method: patient 4 lesion2 is the structure with $\lambda_\mathrm{bio}$ at the bound and 59% of the TIA extrapolated (notebook 02), and patient 6 left kidney has the weakest voxel agreement with MIM (notebook 03).

**Effect of excluding Devasia patient 4.** Only the literature uptake-washout shape changes; the other methods do not use it.

| Scan | median \|error\| %, before | after | max \|error\| %, before | after | in reference 68%, before | after |
|---|---|---|---|---|---|---|
| ~4 h | 17.9 | 19.9 | 28.8 | 24.9 | 0/4 | 0/4 |
| ~30 h | 4.1 | 9.2 | 6.0 | 10.6 | 3/4 | 1/4 |
| ~100 h | 3.5 | 6.0 | 10.6 | 7.9 | 3/4 | 2/4 |
| last | 6.8 | 8.0 | 12.6 | 14.2 | 2/4 | 1/4 |

The earlier values were optimistic, because patient 4 was partly predicted from itself.

**Implications for the project.** A single SPECT at about 4 days recovers organ TIAs within a few percent even with a fixed literature half-life or the Hänscheid formula, so the personalisation step (Bayesian update, notebook 07) should be designed around a scan at about 96-100 h, and its added value has to be shown against these simple baselines rather than against no correction. Lesion kinetics need individual information: population half-lives transfer poorly between patients. Limitations: two patients; the literature half-lives come from planar imaging of the main component, mixed DOTATATE/DOTATOC (Hänscheid), and the only literature uptake-washout shape is for kidneys.

## 6. Saving

Outputs in `data/processed/deepblue/kinetics/`:
- `stp_tia.csv`: one row per patient, structure, scan, method and source (measured activity, STP TIA, reference TIA and 68% interval, error, inside-68% flag, $t / T_\mathrm{eff}$ of the reference);
- `stp_population_kinetics.csv`: population values used, per patient and source;
- `stp_summary_by_scan.csv`, `stp_summary_late_by_type.csv`;
- `stp_error_vs_time.png`.

In [ ]:
stp.drop(columns="label").to_csv(KIN / "stp_tia.csv", index=False)
pop.to_csv(KIN / "stp_population_kinetics.csv", index=False)
by_scan.to_csv(KIN / "stp_summary_by_scan.csv")
by_type.to_csv(KIN / "stp_summary_late_by_type.csv")
print(len(stp), "rows saved")